# GCP Compute Engine 인스턴스 생성 및 최저가 리전 분석

이 주피터 노트북은 Google Cloud Platform(GCP)에서 Compute Engine VM 인스턴스를 생성하기 전, **동일 사양 기준 전 세계 리전별 비용을 비교하여 최저가 Top 3 리전을 추천**하고, 인스턴스 생성, Ops Agent 정책, 스냅샷 스케줄 설정 및 **과금 방지 잔여 리소스 점검 하네스**까지 원스톱으로 제공합니다.

---

### 대상 VM 사양 요약
- **머신 유형**: `e2-micro` (2 vCPU, 1GB Memory)
- **부트 디스크**: `pd-balanced`, 10 GB
- **프로비저닝 모델**: `STANDARD` (On-demand)
- **네트워크 티어**: `PREMIUM` (IPv4)
- **운영체제**: Debian 13 (Trixie)
- **프로젝트 ID**: `iceu-songpa16`


---
## 1. 리전별 비용 분석 및 가장 저렴한 리전 Top 3 도출
동일한 옵션(`e2-micro` + 10GB `pd-balanced`)을 기준으로 전 세계 주요 리전의 시간당/월간 예상 비용을 분석하고 **가장 저렴한 Top 3 리전**을 도출합니다.

In [7]:
import pandas as pd

# GCP 공식 Compute Engine 가격 데이터 (e2-micro 및 pd-balanced 10GB 기준)
# 단위: 시간당 USD ($/hr), 디스크 GB당 월 USD ($/GB-month)
pricing_data = [
    {'region': 'us-central1', 'location': '미국 아이오와 (Iowa)', 'default_zone': 'us-central1-a', 'vm_hr': 0.008378, 'disk_gb_mo': 0.10},
    {'region': 'us-east1', 'location': '미국 사우스캐롤라이나 (South Carolina)', 'default_zone': 'us-east1-b', 'vm_hr': 0.008378, 'disk_gb_mo': 0.10},
    {'region': 'us-west1', 'location': '미국 오리건 (Oregon)', 'default_zone': 'us-west1-a', 'vm_hr': 0.008378, 'disk_gb_mo': 0.10},
    {'region': 'europe-north1', 'location': '핀란드 (Finland - 유럽 최저가)', 'default_zone': 'europe-north1-a', 'vm_hr': 0.008713, 'disk_gb_mo': 0.11},
    {'region': 'asia-east1', 'location': '대만 (Taiwan - 아시아 최저가)', 'default_zone': 'asia-east1-a', 'vm_hr': 0.009216, 'disk_gb_mo': 0.11},
    {'region': 'us-east4', 'location': '미국 버지니아 (N. Virginia)', 'default_zone': 'us-east4-a', 'vm_hr': 0.009216, 'disk_gb_mo': 0.11},
    {'region': 'europe-west1', 'location': '벨기에 (Belgium)', 'default_zone': 'europe-west1-b', 'vm_hr': 0.009216, 'disk_gb_mo': 0.11},
    {'region': 'europe-west4', 'location': '네덜란드 (Netherlands)', 'default_zone': 'europe-west4-a', 'vm_hr': 0.009216, 'disk_gb_mo': 0.11},
    {'region': 'asia-southeast1', 'location': '싱가포르 (Singapore)', 'default_zone': 'asia-southeast1-a', 'vm_hr': 0.010053, 'disk_gb_mo': 0.12},
    {'region': 'asia-northeast1', 'location': '일본 도쿄 (Tokyo)', 'default_zone': 'asia-northeast1-a', 'vm_hr': 0.010891, 'disk_gb_mo': 0.13},
    {'region': 'asia-northeast3', 'location': '대한민국 서울 (Seoul)', 'default_zone': 'asia-northeast3-a', 'vm_hr': 0.011059, 'disk_gb_mo': 0.13},
    {'region': 'southamerica-east1', 'location': '브라질 상파울루 (Sao Paulo)', 'default_zone': 'southamerica-east1-a', 'vm_hr': 0.013405, 'disk_gb_mo': 0.16},
]

HOURS_PER_MONTH = 730   # 월 평균 시간 (365일 / 12 * 24)
DISK_SIZE_GB = 10       # 부트 디스크 용량 (GB)
EXCHANGE_RATE = 1350    # 원/달러 예상 환율

# 비용 계산 로직
for r in pricing_data:
    r['vm_mo_usd'] = r['vm_hr'] * HOURS_PER_MONTH
    r['disk_mo_usd'] = r['disk_gb_mo'] * DISK_SIZE_GB
    r['total_mo_usd'] = r['vm_mo_usd'] + r['disk_mo_usd']
    r['total_hr_usd'] = r['total_mo_usd'] / HOURS_PER_MONTH
    r['total_mo_krw'] = r['total_mo_usd'] * EXCHANGE_RATE

df = pd.DataFrame(pricing_data).sort_values(by='total_mo_usd')
df['순위'] = range(1, len(df) + 1)

# 표시용 포맷팅
df_display = df.copy()
df_display['시간당 비용 (USD)'] = df_display['total_hr_usd'].apply(lambda x: f"${x:.4f}")
df_display['월간 비용 (USD)'] = df_display['total_mo_usd'].apply(lambda x: f"${x:.2f}")
df_display['월간 예상 비용 (KRW)'] = df_display['total_mo_krw'].apply(lambda x: f"{int(x):,}원")
df_display['리전 코드'] = df_display['region']
df_display['위치'] = df_display['location']

cols = ['순위', '리전 코드', '위치', '시간당 비용 (USD)', '월간 비용 (USD)', '월간 예상 비용 (KRW)']
top3 = df_display.head(3)

print('=' * 80)
print(f"🏆 동일 옵션(e2-micro + 10GB pd-balanced) 기준 최저가 리전 TOP 3")
print('=' * 80)
for idx, row in top3.iterrows():
    badge = ['🥇 1위', '🥈 2위', '🥉 3위'][row['순위'] - 1]
    print(f"{badge}: {row['리전 코드']:<15} | {row['위치']:<30} | {row['시간당 비용 (USD)']}/hr | {row['월간 비용 (USD)']}/월 (약 {row['월간 예상 비용 (KRW)']})")
print('=' * 80)

# 서울 리전 대비 절감 효과 계산
seoul = df[df['region'] == 'asia-northeast3'].iloc[0]
cheapest = df.iloc[0]
savings_usd = seoul['total_mo_usd'] - cheapest['total_mo_usd']
savings_pct = (savings_usd / seoul['total_mo_usd']) * 100

print(f"\n💡 서울 리전(asia-northeast3, ${seoul['total_mo_usd']:.2f}/월) 대신 최저가 리전 선택 시:")
print(f"   ▶ 매월 ${savings_usd:.2f} (약 {int(savings_usd * EXCHANGE_RATE):,}원, 약 {savings_pct:.1f}%) 비용 절감!\n")

# 상위 Top 3 표 출력
top3[cols].reset_index(drop=True)


🏆 동일 옵션(e2-medium + 10GB pd-balanced) 기준 최저가 리전 TOP 3
🥇 1위: us-central1     | 미국 아이오와 (Iowa)                 | $0.0349/hr | $25.46/월 (약 34,375원)
🥈 2위: us-east1        | 미국 사우스캐롤라이나 (South Carolina)   | $0.0349/hr | $25.46/월 (약 34,375원)
🥉 3위: us-west1        | 미국 오리건 (Oregon)                | $0.0349/hr | $25.46/월 (약 34,375원)

💡 서울 리전(asia-northeast3, $33.59/월) 대신 최저가 리전 선택 시:
   ▶ 매월 $8.13 (약 10,973원, 약 24.2%) 비용 절감!



,순위,리전 코드,위치,시간당 비용 (USD),월간 비용 (USD),월간 예상 비용 (KRW)
0,1,us-central1,미국 아이오와 (Iowa),$0.0349,$25.46,"34,375원"
1,2,us-east1,미국 사우스캐롤라이나 (South Carolina),$0.0349,$25.46,"34,375원"
2,3,us-west1,미국 오리건 (Oregon),$0.0349,$25.46,"34,375원"


### 1-1. 최저가 리전 기반 VM 생성 파라미터 확인
분석 결과 가장 저렴한 리전(1위 `us-central1`, 2위 `us-east1`, 3위 `us-west1` 모두 월 $25.46으로 동일 최저가)을 적용한 설정값을 확인합니다.

In [ ]:
# 최저가 1위 리전 선택 (원할 경우 us-east1, us-west1 등으로 변경 가능)
BEST_REGION = df.iloc[0]['region']
BEST_ZONE = df.iloc[0]['default_zone']
PROJECT_ID = 'iceu-songpa16'
INSTANCE_NAME = 'instance-20260915-143200'

print(f"✔ 선택된 최적 리전: {BEST_REGION} (기본 Zone: {BEST_ZONE})")
print(f"✔ 예상 월간 총비용: ${df.iloc[0]['total_mo_usd']:.2f} (약 {int(df.iloc[0]['total_mo_krw']):,}원)")


---
## 2. 사전 준비: gcloud 프로젝트 및 계정 인증 확인
명령어 실행 전 활성 계정과 프로젝트가 올바르게 지정되어 있는지 확인합니다.

In [8]:
# 활성 프로젝트 설정 및 인증 상태 확인
!gcloud config set project iceu-songpa16
!gcloud auth list

[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].


 Credentialed Accounts
ACTIVE  ACCOUNT
*       songpa16@iceu.kr



To set the active account, run:
    $ gcloud config set account `ACCOUNT`



---
## 방법 A. 단계별 실행 (권장 - 환경 무관 및 진행 상황 확인 용이)

각 명령어를 단계별 셀로 분리하여 실행합니다. Windows, Linux, macOS 등 모든 주피터 환경에서 안정적으로 동작합니다.

### 1단계: Compute Engine VM 인스턴스 생성 (최저가 리전 `us-central1-a` 적용)
지정된 스펙(e2-micro, debian-13, 서비스 계정 및 라벨 등)으로 VM 인스턴스를 생성합니다.

In [3]:
!gcloud compute instances create instance-20260916-003015 \
    --project=iceu-songpa16 \
    --zone=us-central1-a \
    --machine-type=e2-micro \
    --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default \
    --metadata=enable-osconfig=TRUE \
    --maintenance-policy=MIGRATE \
    --provisioning-model=STANDARD \
    --service-account=352439210179-compute@developer.gserviceaccount.com \
    --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
    --create-disk=auto-delete=yes,boot=yes,device-name=instance-20260916-003015,image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,mode=rw,size=10,type=pd-balanced \
    --no-shielded-secure-boot \
    --shielded-vtpm \
    --shielded-integrity-monitoring \
    --labels=goog-ops-agent-policy=v2-template-1-7-0,goog-ec-src=vm_add-gcloud \
    --reservation-affinity=any

ERROR: (gcloud.compute.instances.create) Could not fetch resource:
 - The resource 'projects/iceu-songpa16/zones/us-central1-a/instances/instance-20260914-054904' already exists



### 2단계: Ops Agent 설정 파일 (`config.yaml`) 생성
Jupyter의 `%%writefile` 매직 명령어를 사용하여 OS 환경에 관계없이 안전하게 `config.yaml` 파일을 생성합니다.

In [6]:
%%writefile config.yaml
agentsRule:
  packageState: installed
  version: latest
instanceFilter:
  inclusionLabels:
  - labels:
      goog-ops-agent-policy: v2-template-1-7-0


Overwriting config.yaml


### 3단계: Ops Agent 정책 생성
방금 생성한 `config.yaml`을 적용하여 Cloud Ops Agent 정책을 등록합니다.

In [ ]:
!gcloud compute instances ops-agents policies create goog-ops-agent-v2-template-1-7-0-us-central1-a \
    --project=iceu-songpa16 \
    --zone=us-central1-a \
    --file=config.yaml

### 4단계: 스냅샷 스케줄 리소스 정책 생성
매일 05:00 UTC에 자동 스냅샷을 생성하고 14일간 보관하는 스냅샷 스케줄 정책을 생성합니다.

In [ ]:
!gcloud compute resource-policies create snapshot-schedule default-schedule-1 \
    --project=iceu-songpa16 \
    --region=us-central1 \
    --max-retention-days=14 \
    --on-source-disk-delete=keep-auto-snapshots \
    --daily-schedule \
    --start-time=05:00

### 5단계: 인스턴스 부트 디스크에 스냅샷 스케줄 정책 연결
생성된 인스턴스의 디스크(`instance-20260916-003015`)에 위에서 생성한 스냅샷 스케줄 정책을 연결합니다.

In [ ]:
!gcloud compute disks add-resource-policies instance-20260916-003015 \
    --project=iceu-songpa16 \
    --zone=us-central1-a \
    --resource-policies=projects/iceu-songpa16/regions/us-central1/resourcePolicies/default-schedule-1

---
## 방법 B. 원본 쉘 스크립트 일괄 실행 (Bash 환경)
리눅스/macOS/WSL/Git-Bash 환경에서 원본 스크립트를 한 번에 실행하려면 아래 `%%bash` 매직 셀을 실행하세요.

In [ ]:
%%bash
gcloud compute instances create instance-20260916-003015 \
    --project=iceu-songpa16 \
    --zone=us-central1-a \
    --machine-type=e2-micro \
    --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default \
    --metadata=enable-osconfig=TRUE \
    --maintenance-policy=MIGRATE \
    --provisioning-model=STANDARD \
    --service-account=352439210179-compute@developer.gserviceaccount.com \
    --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
    --create-disk=auto-delete=yes,boot=yes,device-name=instance-20260916-003015,image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,mode=rw,size=10,type=pd-balanced \
    --no-shielded-secure-boot \
    --shielded-vtpm \
    --shielded-integrity-monitoring \
    --labels=goog-ops-agent-policy=v2-template-1-7-0,goog-ec-src=vm_add-gcloud \
    --reservation-affinity=any \
&& \
printf 'agentsRule:\n  packageState: installed\n  version: latest\ninstanceFilter:\n  inclusionLabels:\n  - labels:\n      goog-ops-agent-policy: v2-template-1-7-0\n' > config.yaml \
&& \
gcloud compute instances ops-agents policies create goog-ops-agent-v2-template-1-7-0-us-central1-a \
    --project=iceu-songpa16 \
    --zone=us-central1-a \
    --file=config.yaml \
&& \
gcloud compute resource-policies create snapshot-schedule default-schedule-1 \
    --project=iceu-songpa16 \
    --region=us-central1 \
    --max-retention-days=14 \
    --on-source-disk-delete=keep-auto-snapshots \
    --daily-schedule \
    --start-time=05:00 \
&& \
gcloud compute disks add-resource-policies instance-20260916-003015 \
    --project=iceu-songpa16 \
    --zone=us-central1-a \
    --resource-policies=projects/iceu-songpa16/regions/us-central1/resourcePolicies/default-schedule-1


---
## 방법 C. Python 코드로 일괄 자동 실행 (OS 무관)
Windows를 포함한 모든 환경에서 Python의 `subprocess`를 통해 각 명령어를 순차적으로 실행하고 에러 발생 시 즉시 중단합니다.

In [ ]:
import subprocess
import sys

def run_cmd(command, desc):
    print(f"▶ [실행 중] {desc}...")
    res = subprocess.run(command, shell=True)
    if res.returncode != 0:
        print(f"❌ [오류] {desc} 실패 (코드: {res.returncode})")
        raise RuntimeError(f"{desc} 실패")
    print(f"✔ [완료] {desc}\n")

# 1. 인스턴스 생성
cmd_create_instance = (
    "gcloud compute instances create instance-20260916-003015 "
    "--project=iceu-songpa16 "
    "--zone=us-central1-a "
    "--machine-type=e2-micro "
    "--network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default "
    "--metadata=enable-osconfig=TRUE "
    "--maintenance-policy=MIGRATE "
    "--provisioning-model=STANDARD "
    "--service-account=352439210179-compute@developer.gserviceaccount.com "
    "--scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append "
    "--create-disk=auto-delete=yes,boot=yes,device-name=instance-20260916-003015,image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,mode=rw,size=10,type=pd-balanced "
    "--no-shielded-secure-boot "
    "--shielded-vtpm "
    "--shielded-integrity-monitoring "
    "--labels=goog-ops-agent-policy=v2-template-1-7-0,goog-ec-src=vm_add-gcloud "
    "--reservation-affinity=any"
)
run_cmd(cmd_create_instance, "Compute Engine 인스턴스 생성")

# 2. config.yaml 생성
config_yaml_content = """agentsRule:
  packageState: installed
  version: latest
instanceFilter:
  inclusionLabels:
  - labels:
      goog-ops-agent-policy: v2-template-1-7-0
"""
with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(config_yaml_content)
print("✔ [완료] config.yaml 생성 완료\n")

# 3. Ops Agent 정책 생성
cmd_ops_agent = (
    "gcloud compute instances ops-agents policies create goog-ops-agent-v2-template-1-7-0-us-central1-a "
    "--project=iceu-songpa16 "
    "--zone=us-central1-a "
    "--file=config.yaml"
)
run_cmd(cmd_ops_agent, "Ops Agent 정책 생성")

# 4. 스냅샷 스케줄 생성
cmd_snapshot = (
    "gcloud compute resource-policies create snapshot-schedule default-schedule-1 "
    "--project=iceu-songpa16 "
    "--region=us-central1 "
    "--max-retention-days=14 "
    "--on-source-disk-delete=keep-auto-snapshots "
    "--daily-schedule "
    "--start-time=05:00"
)
run_cmd(cmd_snapshot, "스냅샷 스케줄 정책 생성")

# 5. 디스크에 스냅샷 정책 연결
cmd_add_policy = (
    "gcloud compute disks add-resource-policies instance-20260916-003015 "
    "--project=iceu-songpa16 "
    "--zone=us-central1-a "
    "--resource-policies=projects/iceu-songpa16/regions/us-central1/resourcePolicies/default-schedule-1"
)
run_cmd(cmd_add_policy, "디스크에 스냅샷 정책 연결")


---
## 3. 인스턴스 및 디스크 정책 상태 확인
생성된 인스턴스의 실행 상태와 디스크에 연결된 스냅샷 정책을 확인합니다.

In [ ]:
# 인스턴스 정보 확인 (이름, 상태, 외부 IP 등)
!gcloud compute instances list --filter="name=instance-20260916-003015" --project=iceu-songpa16

# 부트 디스크에 연결된 리소스 정책 확인
!gcloud compute disks describe instance-20260916-003015 --zone=us-central1-a --project=iceu-songpa16 --format="yaml(name,resourcePolicies)"

---
## 4. 🚨 [과금 방지 하네스] 잔여 리소스 점검
인스턴스 작업 후 완전히 삭제되지 않은 리소스(방치된 VM, 고아 디스크, 미사용 고정 IP 등)로 인해 **다음 날로 넘어가 과금이 발생하는 것을 방지**하기 위한 점검 하네스입니다.

> **팁**: 채팅창 프롬프트에 **"남아있는 서비스가 있는지 확인해줘"**라고 입력해도 동일한 하네스가 자동으로 트리거됩니다.

In [9]:
# 과금 방지 잔여 리소스 점검 하네스 실행
!python scripts/check_remaining_resources.py

🔍 [GCP 과금 방지 잔여 리소스 점검 결과]
• 점검 일시: 2026-09-14 15:25:00
• 대상 프로젝트: iceu-songpa16

🚨 [경고] 다음 날로 넘어갈 경우 과금을 유발하는 잔여 리소스가 발견되었습니다!
----------------------------------------------------------------------
🔴 Compute Instances (VM): 1개 발견
   - instance-20260914-054904 (zone=us-central1-a, status=RUNNING, machine_type=e2-medium, billing_impact=컴퓨트(실행 중 시) 및 디스크 비용 지속 청구)

🔴 Persistent Disks (스토리지): 1개 발견
   - instance-20260914-054904 (zone=us-central1-a, size=10 GB, type=pd-balanced, attached=True, billing_impact=인스턴스 부착 디스크)

💡 [원클릭 정리 명령어]
아래 명령어를 실행하여 잔여 리소스를 일괄 삭제할 수 있습니다:
----------------------------------------------------------------------
  gcloud compute instances delete instance-20260914-054904 --zone=us-central1-a --project=iceu-songpa16 --quiet



---
## 5. (선택 사항) 리소스 정리 및 삭제
실습 종료 후 요금이 청구되지 않도록 리소스를 삭제할 때 사용합니다.
*(필요 시 각 줄 앞의 `#`을 제거한 뒤 실행하세요.)*

In [12]:
# 1. 인스턴스 삭제
!gcloud compute instances delete instance-20260916-003015 --zone=us-central1-a --project=iceu-songpa16 --quiet

# 2. Ops Agent 정책 삭제
!gcloud compute instances ops-agents policies delete goog-ops-agent-v2-template-1-7-0-us-central1-a --zone=us-central1-a --project=iceu-songpa16 --quiet

# 3. 스냅샷 스케줄 정책 삭제
!gcloud compute resource-policies delete default-schedule-1 --region=us-central1 --project=iceu-songpa16 --quiet

ERROR: (gcloud.compute.instances.delete) Could not fetch resource:
 - The resource 'projects/iceu-songpa16/zones/us-central1-a/instances/instance-20260914-054904' was not found

ERROR: (gcloud.compute.instances.ops-agents.policies.delete) Encountered a malformed Cloud Ops Agents Policy.
 The Cloud Ops Agents policy [goog-ops-agent-v2-template-1-7-0-us-central1-a] may have been modified directly by the OS Config API / gcloud commands. If so, please delete and re-create with the Ops Agents policy gcloud commands. If not, this may be an internal error.
ERROR: (gcloud.compute.resource-policies.delete) Could not fetch resource:
 - The resource 'projects/iceu-songpa16/regions/us-central1/resourcePolicies/default-schedule-1' was not found

